# Ứng dụng 2: Dự đoán trang tiếp theo trên Wikispeedia — GSP và PrefixSpan

Notebook này chạy lại toàn bộ thực nghiệm của Ứng dụng 2:

1. Đọc archive gốc của SNAP, khôi phục thao tác quay lại `<`, giữ đường đi có ít nhất 2 lượt, xáo trộn một lần bằng seed cố định.
2. Kiểm tra tính đúng của hai bản cài đặt (ví dụ tính tay, trường hợp biên, so với cách đếm vét cạn).
3. Đo thời gian và bộ nhớ của GSP và PrefixSpan theo ngưỡng r và theo số chuỗi.
4. Đánh giá chức năng gợi ý trên tập test so với baseline trang phổ biến.

Toàn bộ code chỉ dùng thư viện chuẩn của Python (riêng biểu đồ dùng matplotlib). Các ô `%%writefile` ghi ra đúng các tệp `.py` đã dùng khi chạy trên máy cục bộ.

**Chạy trên Kaggle:** cần một trong hai cách để có dữ liệu:
- bật *Internet* trong phần Settings của notebook để tải archive từ SNAP; hoặc
- thêm một Kaggle Dataset chứa tệp `wikispeedia_paths-and-graph.tar.gz` (notebook tự tìm trong `/kaggle/input`).

Thời gian chạy toàn bộ trên máy cục bộ (Intel i7-14650HX) là vài phút; trên Kaggle có thể lâu hơn vì CPU khác.

## 1. Tìm hoặc tải dữ liệu

Archive gốc: [SNAP — Wikispeedia navigation paths](https://snap.stanford.edu/data/wikispeedia.html). Mã SHA-256 của bản đã dùng được kiểm tra để bảo đảm cùng dữ liệu.

In [ ]:
import hashlib, os, sys, urllib.request
from pathlib import Path

ARCHIVE_NAME = "wikispeedia_paths-and-graph.tar.gz"
ARCHIVE_SHA256 = "97697096f5d2dcb77aa69e3992305c6c561de89edb9fb10b5ad9feaf8ba534d5"
SNAP_URL = "https://snap.stanford.edu/data/wikispeedia/" + ARCHIVE_NAME
WORK = Path.cwd()

candidates = list(Path("/kaggle/input").rglob(ARCHIVE_NAME)) if Path("/kaggle/input").exists() else []
candidates += [Path(p) for p in os.environ.get("WIKISPEEDIA_ARCHIVE", "").split(os.pathsep) if p]
candidates += [WORK / ARCHIVE_NAME]
archive = next((p for p in candidates if p.exists()), None)
if archive is None:
    archive = WORK / ARCHIVE_NAME
    print("Tải từ", SNAP_URL)
    urllib.request.urlretrieve(SNAP_URL, archive)
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
print(archive, digest)
assert digest == ARCHIVE_SHA256, "Archive khác bản đã dùng trong bài"

## 2. Thư viện của bài: `spm_wiki.py`

Gồm: đọc và khôi phục đường đi, ngưỡng `min_sup = ⌈r·|D|⌉` tính bằng phân số chính xác, GSP (nối + cắt tỉa + cây tiền tố, khớp sớm nhất), PrefixSpan (pseudo-projection một vị trí mỗi chuỗi), cách đếm vét cạn để kiểm tra, hàm đo một lượt và bộ gợi ý theo quy tắc mục 4.7.

In [ ]:
%%writefile spm_wiki.py
"""GSP và PrefixSpan tự cài đặt cho Ứng dụng 2: dự đoán trang tiếp theo trên Wikispeedia.

Mỗi event là đúng một trang, nên một chuỗi là tuple mã trang và mẫu cũng là tuple mã trang.
Chỉ có mở rộng chuỗi (S-extension); không có mẫu dạng <(a b)>.

Dùng như thư viện (import) hoặc dòng lệnh:
    python -I spm_wiki.py prepare --archive <tar.gz> --outdir <thư mục>
    python -I spm_wiki.py bench --algo gsp|prefixspan --data <sequences.tsv> --n N --r 0.002
                                --maxlen 4 --mode time|mem --out <patterns.tsv> --json <run.json>
Chỉ dùng thư viện chuẩn của Python.
"""
import argparse
import hashlib
import json
import math
import random
import sys
import tarfile
import time
import tracemalloc
from collections import Counter, defaultdict
from fractions import Fraction
from itertools import combinations
from pathlib import Path
from urllib.parse import unquote

SEED = 20261101
MAX_LEN = 4


# ---------------------------------------------------------------- dữ liệu

def read_wikispeedia(archive_path):
    """Đọc archive SNAP, khôi phục thao tác quay lại '<' bằng ngăn xếp.

    Trả về (names, records, stats). records là list (sid, seq, back): seq là tuple mã trang theo
    articles.tsv, back[i] = 1 nếu lượt ghé thứ i đến từ thao tác quay lại.
    """
    with tarfile.open(archive_path, "r:gz") as tar:
        members = {m.name.rsplit("/", 1)[-1]: m for m in tar.getmembers() if m.isfile()}

        def lines(filename):
            for raw in tar.extractfile(members[filename]):
                text = raw.decode("utf-8").rstrip("\r\n")
                if text and not text.startswith("#"):
                    yield text

        articles = list(lines("articles.tsv"))
        ids = {name: i for i, name in enumerate(articles)}
        names = [unquote(name) for name in articles]
        records, stats = [], Counter()
        for kind, n_fields in (("finished", 5), ("unfinished", 6)):
            for row, line in enumerate(lines(f"paths_{kind}.tsv")):
                fields = line.split("\t")
                assert len(fields) == n_fields, fields
                stats[f"raw_{kind}"] += 1
                stack, seq, back, ok = [], [], [], True
                for token in fields[3].split(";"):
                    if token == "<":
                        if len(stack) < 2:
                            ok = False
                            break
                        stack.pop()
                        seq.append(stack[-1])
                        back.append(1)
                    elif token in ids:
                        stack.append(ids[token])
                        seq.append(ids[token])
                        back.append(0)
                    else:
                        ok = False
                        break
                if not ok:
                    stats["unresolvable"] += 1
                    continue
                records.append((f"{kind[0].upper()}{row}", tuple(seq), tuple(back)))
    return names, records, stats


def prepare(archive_path, outdir, seed=SEED):
    """Giữ chuỗi có ít nhất 2 lượt, xáo trộn một lần bằng seed cố định, ghi ra outdir."""
    names, records, stats = read_wikispeedia(archive_path)
    kept = [r for r in records if len(r[1]) >= 2]
    random.Random(seed).shuffle(kept)
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    rows = ["sid\tpages\tback"]
    rows += [f"{sid}\t{' '.join(map(str, seq))}\t{''.join(map(str, back))}" for sid, seq, back in kept]
    (outdir / "sequences.tsv").write_bytes(("\n".join(rows) + "\n").encode("utf-8"))
    (outdir / "articles.tsv").write_bytes(
        ("id\tname\n" + "".join(f"{i}\t{n}\n" for i, n in enumerate(names))).encode("utf-8"))
    lengths = [len(r[1]) for r in kept]
    info = {
        "seed": seed,
        "raw_paths": len(records) + stats["unresolvable"],
        "raw_finished": stats["raw_finished"],
        "raw_unfinished": stats["raw_unfinished"],
        "unresolvable": stats["unresolvable"],
        "dropped_len1": len(records) - len(kept),
        "kept": len(kept),
        "visits": sum(lengths),
        "back_visits": sum(sum(r[2]) for r in kept),
        "distinct_pages": len({p for r in kept for p in r[1]}),
        "mean_len": sum(lengths) / len(lengths),
        "max_len": max(lengths),
        "n_train": math.floor(Fraction(4, 5) * len(kept)),
    }
    info["n_test"] = info["kept"] - info["n_train"]
    (outdir / "prepare_info.json").write_bytes(json.dumps(info, indent=2).encode("utf-8"))
    return info


def read_sequences(path, n=None, start=0, with_back=False):
    """Đọc n chuỗi kể từ dòng start (theo thứ tự đã xáo trộn) của sequences.tsv."""
    seqs, backs, sids = [], [], []
    with open(path, encoding="utf-8") as f:
        next(f)
        for i, line in enumerate(f):
            if i < start:
                continue
            if n is not None and len(seqs) >= n:
                break
            sid, pages, back = line.rstrip("\n").split("\t")
            seqs.append(tuple(map(int, pages.split())))
            if with_back:
                backs.append(tuple(map(int, back)))
                sids.append(sid)
    return (seqs, backs, sids) if with_back else seqs


def read_names(path):
    with open(path, encoding="utf-8") as f:
        next(f)
        return [line.rstrip("\n").split("\t", 1)[1] for line in f]


# ---------------------------------------------------------------- ngưỡng

def min_sup_from_ratio(r, n_sequences):
    """min_sup = ceil(r * |D|) tính bằng phân số chính xác. r nhận chuỗi '0.002' hoặc số thực."""
    return math.ceil(Fraction(str(r)) * n_sequences)


def contains(pattern, seq):
    """pattern ⊑ seq theo khớp tham lam từ trái sang."""
    it = iter(seq)
    return all(any(x == y for y in it) for x in pattern)


# ---------------------------------------------------------------- GSP

def gsp(db, min_sup, max_len=MAX_LEN):
    """GSP (Srikant & Agrawal 1996) cho event một item, không ràng buộc gap/window.

    Mỗi mức k: sinh C_k bằng nối + cắt tỉa, đặt C_k vào cây tiền tố, quét toàn bộ db một lần để đếm.
    Trả về (patterns, stats) với patterns: dict tuple -> sup.
    """
    stats = {"candidates": {}, "frequent": {}}
    item_count = Counter()
    for s in db:
        item_count.update(set(s))
    level = {(x,): c for x, c in item_count.items() if c >= min_sup}
    stats["candidates"][1] = len(item_count)
    stats["frequent"][1] = len(level)
    patterns = dict(level)
    k = 2
    while level and k <= max_len:
        candidates = _gsp_generate(sorted(level), k)
        stats["candidates"][k] = len(candidates)
        if not candidates:
            break
        counts = _gsp_count(db, candidates, k)
        level = {c: n for c, n in zip(candidates, counts) if n >= min_sup}
        stats["frequent"][k] = len(level)
        patterns.update(level)
        del candidates, counts
        k += 1
    return patterns, stats


def _gsp_generate(prev, k):
    """Pha nối + pha cắt tỉa. prev là list mẫu độ dài k-1 đã sắp xếp."""
    if k == 2:
        items = [p[0] for p in prev]
        return [(a, b) for a in items for b in items]
    by_prefix = defaultdict(list)
    for s in prev:
        by_prefix[s[:-1]].append(s[-1])
    prev_set = set(prev)
    out = []
    for s1 in prev:
        for y in by_prefix.get(s1[1:], ()):
            c = s1 + (y,)
            # bỏ vị trí 0 cho s2 và vị trí k-1 cho s1, đều thuộc prev; chỉ cần kiểm tra vị trí giữa
            if all(c[:i] + c[i + 1:] in prev_set for i in range(1, k - 1)):
                out.append(c)
    return out


def _gsp_count(db, candidates, k):
    """Đếm support bằng cây tiền tố (thay hash tree của paper) và khớp sớm nhất."""
    trie = {}
    for idx, c in enumerate(candidates):
        node = trie
        for x in c[:-1]:
            node = node.setdefault(x, {})
        node[c[-1]] = idx
    counts = [0] * len(candidates)
    for s in db:
        _gsp_visit(trie, s, 0, k, counts)
    return counts


def _gsp_visit(node, s, start, depth_left, counts):
    seen = set()
    for j in range(start, len(s)):
        x = s[j]
        if x in seen:
            continue
        seen.add(x)
        child = node.get(x)
        if child is None:
            continue
        if depth_left == 1:
            counts[child] += 1
        else:
            _gsp_visit(child, s, j + 1, depth_left - 1, counts)


# ---------------------------------------------------------------- PrefixSpan

def prefixspan(db, min_sup, max_len=MAX_LEN):
    """PrefixSpan (Pei và cộng sự 2004) với pseudo-projection (mã chuỗi, vị trí bắt đầu hậu tố).

    Mỗi chuỗi trong dữ liệu chiếu giữ một vị trí: ngay sau lần khớp sớm nhất của tiền tố.
    Trả về (patterns, stats).
    """
    stats = {"projected_dbs": 0, "projected_entries": 0}
    patterns = {}
    root = [(sid, 0) for sid in range(len(db))]
    _ps_grow((), root, db, min_sup, max_len, patterns, stats)
    return patterns, stats


def _ps_grow(prefix, projected, db, min_sup, max_len, patterns, stats):
    count = Counter()
    for sid, start in projected:
        s = db[sid]
        if start < len(s):
            count.update(set(s[start:]))
    frequent = [x for x, c in count.items() if c >= min_sup]
    if not frequent:
        return
    for x in frequent:
        patterns[prefix + (x,)] = count[x]
    if len(prefix) + 1 >= max_len:
        return
    wanted = set(frequent)
    children = defaultdict(list)
    for sid, start in projected:
        s = db[sid]
        seen = set()
        for j in range(start, len(s)):
            x = s[j]
            if x in wanted and x not in seen:
                seen.add(x)
                children[x].append((sid, j + 1))
    del count, wanted
    stats["projected_dbs"] += len(children)
    stats["projected_entries"] += sum(map(len, children.values()))
    for x in frequent:
        _ps_grow(prefix + (x,), children.pop(x), db, min_sup, max_len, patterns, stats)


ALGORITHMS = {"gsp": gsp, "prefixspan": prefixspan}


# ---------------------------------------------------------------- kiểm tra

def brute_force(db, min_sup, max_len=MAX_LEN):
    """Liệt kê mọi chuỗi con khác nhau của từng chuỗi; chỉ dùng cho dữ liệu nhỏ."""
    count = Counter()
    for s in db:
        subs = set()
        for k in range(1, min(max_len, len(s)) + 1):
            for idx in combinations(range(len(s)), k):
                subs.add(tuple(s[i] for i in idx))
        count.update(subs)
    return {p: c for p, c in count.items() if c >= min_sup}


def pattern_lines(patterns):
    return [" ".join(map(str, p)) + "\t" + str(c) for p, c in sorted(patterns.items(), key=lambda t: (len(t[0]), t[0]))]


def write_patterns(path, patterns):
    data = ("\n".join(pattern_lines(patterns)) + "\n").encode("utf-8")
    Path(path).write_bytes(data)
    return hashlib.sha256(data).hexdigest()


def read_patterns(path):
    out = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                p, c = line.rstrip("\n").split("\t")
                out[tuple(map(int, p.split()))] = int(c)
    return out


# ---------------------------------------------------------------- đo một lượt

def bench_once(algo, data, n, r, max_len, mode, out):
    """Một lượt đo trong tiến trình hiện tại: đọc -> khai phá -> ghi. Bộ đếm giờ bao ngoài thuật toán."""
    if mode == "mem":
        tracemalloc.start()
    t0 = time.perf_counter()
    db = read_sequences(data, n)
    min_sup = min_sup_from_ratio(r, len(db))
    patterns, stats = ALGORITHMS[algo](db, min_sup, max_len)
    digest = write_patterns(out, patterns)
    seconds = time.perf_counter() - t0
    peak = None
    if mode == "mem":
        peak = tracemalloc.get_traced_memory()[1]
        tracemalloc.stop()
    by_len = Counter(len(p) for p in patterns)
    return {
        "algo": algo, "n": len(db), "r": str(r), "min_sup": min_sup, "max_len": max_len, "mode": mode,
        "seconds": seconds, "peak_bytes": peak, "n_patterns": len(patterns),
        "by_length": {str(k): by_len[k] for k in sorted(by_len)}, "sha256": digest, "stats": stats,
    }


# ---------------------------------------------------------------- gợi ý và đánh giá

MAX_CONTEXT = 3
MIN_CONTEXT_COUNT = 5     # C(h) >= 5
MIN_TRANSITION_COUNT = 2  # C(h,y) >= 2


class Recommender:
    """Quy tắc mục 4.7: ngữ cảnh 3 -> 2 -> 1, lọc bằng mẫu phổ biến, xếp theo C(h,y)/C(h)."""

    def __init__(self, train, patterns, max_context=MAX_CONTEXT,
                 min_context=MIN_CONTEXT_COUNT, min_transition=MIN_TRANSITION_COUNT):
        self.patterns = patterns
        self.max_context = max_context
        self.min_context = min_context
        self.min_transition = min_transition
        self.c_h = Counter()
        self.c_hy = defaultdict(Counter)
        visits = Counter()
        for s in train:
            visits.update(s)
            for m in range(1, max_context + 1):
                for i in range(len(s) - m):
                    h = s[i:i + m]
                    self.c_h[h] += 1
                    self.c_hy[h][s[i + m]] += 1
        self.popular = [p for p, _ in sorted(visits.items(), key=lambda t: (-t[1], t[0]))[:max_context + 10]]

    def explain(self, history):
        """Trả về (context, ranked) với ranked là list dict bằng chứng; ([], []) nếu thiếu bằng chứng."""
        for m in range(min(self.max_context, len(history)), 0, -1):
            h = tuple(history[-m:])
            ch = self.c_h.get(h, 0)
            if ch < self.min_context:
                continue
            ranked = []
            for y, chy in self.c_hy[h].items():
                sup = self.patterns.get(h + (y,))
                if chy >= self.min_transition and sup is not None:
                    ranked.append({"page": y, "c_h": ch, "c_hy": chy, "score": chy / ch, "sup": sup})
            if ranked:
                ranked.sort(key=lambda e: (-e["score"], -e["c_hy"], -e["sup"], e["page"]))
                return h, ranked[:3]
        return (), []

    def pattern_top(self, history, k=3):
        return [e["page"] for e in self.explain(history)[1][:k]]

    def baseline_top(self, history, k=3):
        return [p for p in self.popular if p != history[-1]][:k]

    def filled_top(self, history, k=3):
        out = self.pattern_top(history, k)
        for p in self.baseline_top(history, k + len(out)):
            if len(out) >= k:
                break
            if p not in out:
                out.append(p)
        return out


def test_cases(test_seqs, test_backs):
    """Mọi vị trí t = 1..L-1 của mọi đường đi test: (chỉ số đường đi, t, lịch sử, đáp án, là click tiến)."""
    for i, (s, b) in enumerate(zip(test_seqs, test_backs)):
        for t in range(1, len(s)):
            yield i, t, s[:t], s[t], b[t] == 0


def evaluate(rec, test_seqs, test_backs):
    """Coverage, Hit@1, Hit@3 cho ba phương án; tách toàn bộ ca và ca click tiến."""
    variants = {"baseline": rec.baseline_top, "pattern": rec.pattern_top, "pattern+fill": rec.filled_top}
    groups = {"all": Counter(), "forward": Counter()}
    context_len = Counter()
    for _, _, hist, ans, forward in test_cases(test_seqs, test_backs):
        ctx, ranked = rec.explain(hist)
        context_len[len(ctx)] += 1
        tops = {"pattern": [e["page"] for e in ranked]}
        tops["baseline"] = rec.baseline_top(hist)
        tops["pattern+fill"] = rec.filled_top(hist)
        for g in (["all", "forward"] if forward else ["all"]):
            c = groups[g]
            c["T"] += 1
            covered = bool(tops["pattern"])
            for name in variants:
                top = tops[name]
                c[f"{name}.cov"] += bool(top)
                c[f"{name}.hit1"] += bool(top) and top[0] == ans
                c[f"{name}.hit3"] += ans in top
            if covered:
                c["cov.T"] += 1
                c["cov.pattern.hit3"] += ans in tops["pattern"]
                c["cov.baseline.hit3"] += ans in tops["baseline"]
    return {"groups": {g: dict(c) for g, c in groups.items()}, "context_len": dict(context_len)}


# ---------------------------------------------------------------- dòng lệnh

def main(argv=None):
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("prepare")
    p.add_argument("--archive", required=True)
    p.add_argument("--outdir", required=True)
    b = sub.add_parser("bench")
    b.add_argument("--algo", choices=sorted(ALGORITHMS), required=True)
    b.add_argument("--data", required=True)
    b.add_argument("--n", type=int, required=True)
    b.add_argument("--r", required=True)
    b.add_argument("--maxlen", type=int, default=MAX_LEN)
    b.add_argument("--mode", choices=["time", "mem"], default="time")
    b.add_argument("--out", required=True)
    b.add_argument("--json", required=True)
    args = ap.parse_args(argv)
    if args.cmd == "prepare":
        print(json.dumps(prepare(args.archive, args.outdir), indent=2))
    else:
        res = bench_once(args.algo, args.data, args.n, args.r, args.maxlen, args.mode, args.out)
        Path(args.json).write_bytes(json.dumps(res).encode("utf-8"))
        print(json.dumps({k: res[k] for k in ("algo", "n", "min_sup", "mode", "seconds", "peak_bytes", "n_patterns")}))


if __name__ == "__main__":
    sys.exit(main())

## 3. Chuẩn bị dữ liệu thực nghiệm

Giữ cả đường đi hoàn thành và chưa hoàn thành; khôi phục `<` bằng ngăn xếp; giữ chuỗi có ít nhất 2 lượt; xáo trộn một lần với seed 20261101. Các tập con 5.000 → toàn bộ và phần train/test đều lấy từ cùng thứ tự này.

In [ ]:
import json, subprocess
r = subprocess.run([sys.executable, "spm_wiki.py", "prepare", "--archive", str(archive), "--outdir", "data"],
                   capture_output=True, text=True, encoding="utf-8", check=True)
info = json.loads(r.stdout)
info

## 4. Kiểm tra tính đúng (lớp 1–3)

Lớp 1: ví dụ năm chuỗi phải cho đúng 7 mẫu tính tay. Lớp 2: trường hợp biên. Lớp 3: 600 cơ sở dữ liệu ngẫu nhiên, GSP == PrefixSpan == vét cạn. Lớp 4 (đối chiếu SPMF) cần chương trình SPMF nên chỉ chạy trên máy cục bộ; kết quả lớp 4 nằm trong `ket-qua/kiem_tra_tinh_dung.json` của bộ kết quả giao kèm.

In [ ]:
%%writefile kiem_tra_tinh_dung.py
"""Bốn lớp kiểm tra tính đúng của GSP và PrefixSpan (mục 3.7 của bản nội dung).

Chạy: python -I kiem_tra_tinh_dung.py [--spmf <SPMF.exe>]
Kết quả ghi vào ket-qua/kiem_tra_tinh_dung.json.
"""
import argparse
import json
import random
import subprocess
import sys
import tempfile
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
import spm_wiki as sw  # noqa: E402

A, B, C, D, X = 0, 1, 2, 3, 4


def both(db, min_sup, max_len=sw.MAX_LEN):
    return sw.gsp(db, min_sup, max_len)[0], sw.prefixspan(db, min_sup, max_len)[0]


def layer1():
    """Ví dụ năm chuỗi ở mục 3.3: đúng 7 mẫu và sup đã tính tay."""
    db = [(A, B, C), (A, X, B, C), (A, B, C), (A, B, D), (B, C)]
    min_sup = sw.min_sup_from_ratio("0.4", len(db))
    expected = {(A,): 4, (B,): 5, (C,): 4, (A, B): 4, (A, C): 3, (B, C): 4, (A, B, C): 3}
    g, p = both(db, min_sup)
    gsp_stats = sw.gsp(db, min_sup)[1]
    checks = {
        "min_sup == 2": min_sup == 2,
        "GSP == 7 mẫu tính tay": g == expected,
        "PrefixSpan == 7 mẫu tính tay": p == expected,
        "GSP sinh 9 ứng viên mức 2": gsp_stats["candidates"][2] == 9,
        "GSP sinh 1 ứng viên mức 3": gsp_stats["candidates"][3] == 1,
    }
    return checks


def layer2():
    """Trường hợp biên: trang lặp, khoảng cách, chuỗi trùng, làm tròn ngưỡng, giới hạn độ dài."""
    checks = {}
    g, p = both([(A, B, A)], 1)
    checks["<A,A> có trong A→B→A"] = g.get((A, A)) == 1 == p.get((A, A))
    g, p = both([(A, B)], 1)
    checks["<A,A> không có trong A→B"] = (A, A) not in g and (A, A) not in p
    g, p = both([(A, A, A, A, A)], 1)
    checks["chuỗi lặp chỉ đóng góp 1 lần"] = g == p == {(A,) * k: 1 for k in range(1, 5)}
    g, p = both([(A, X, B, C)], 1)
    checks["mẫu có khoảng cách <A,C>"] = g.get((A, C)) == 1 == p.get((A, C))
    g, p = both([(A, B, C), (A, B, C)], 2)
    checks["đường đi trùng nhau vẫn đếm 2"] = g.get((A, B, C)) == 2 == p.get((A, B, C))
    g, p = both([(A, B, C, D, A, B)], 1, max_len=2)
    checks["giới hạn độ dài 2"] = max(map(len, g)) == 2 == max(map(len, p)) and g == p
    g, p = both([(A, B, C, D, A, B)], 1, max_len=1)
    checks["giới hạn độ dài 1"] = g == p == {(A,): 1, (B,): 1, (C,): 1, (D,): 1}
    g, p = both([(A, B), (C, D)], 3)
    checks["ngưỡng quá cao cho tập rỗng"] = g == p == {}
    checks["ceil(0,07 × 100) = 7 (chuỗi)"] = sw.min_sup_from_ratio("0.07", 100) == 7
    checks["ceil(0,07 × 100) = 7 (số thực)"] = sw.min_sup_from_ratio(0.07, 100) == 7
    checks["ceil(0,002 × 20.000) = 40"] = sw.min_sup_from_ratio("0.002", 20000) == 40
    checks["ceil(0,002 × 56.774) = 114"] = sw.min_sup_from_ratio("0.002", 56774) == 114
    checks["ceil(0,001 × 5.000) = 5"] = sw.min_sup_from_ratio("0.001", 5000) == 5
    return checks


def layer3(trials=600, seed=sw.SEED):
    """So với cách đếm vét cạn trên nhiều cơ sở dữ liệu ngẫu nhiên nhỏ."""
    rng = random.Random(seed)
    failures = []
    total_patterns = 0
    for t in range(trials):
        alphabet = rng.randint(2, 7)
        db = [tuple(rng.randrange(alphabet) for _ in range(rng.randint(1, 10)))
              for _ in range(rng.randint(1, 30))]
        min_sup = rng.randint(1, 6)
        max_len = rng.randint(1, 4)
        ref = sw.brute_force(db, min_sup, max_len)
        g, p = both(db, min_sup, max_len)
        total_patterns += len(ref)
        if not (g == ref == p):
            failures.append({"trial": t, "db": db, "min_sup": min_sup, "max_len": max_len})
    return {f"{trials} CSDL ngẫu nhiên: GSP == PrefixSpan == vét cạn": not failures,
            "_total_reference_patterns": total_patterns, "_failures": failures[:3]}


def layer4(spmf_exe, data, configs=((5000, "0.01"), (5000, "0.005"), (5000, "0.002"), (20000, "0.002"))):
    """Đối chiếu với SPMF PrefixSpan trên tập con Wikispeedia thật (n chuỗi đầu theo thứ tự xáo trộn)."""
    checks, info = {}, {}
    with tempfile.TemporaryDirectory(dir=HERE / "ket-qua") as tmp:
        tmp = Path(tmp)
        for n, r in configs:
            db = sw.read_sequences(data, n)
            inp = tmp / f"spmf_input_{n}.txt"
            # SPMF: mỗi event kết thúc bằng -1, mỗi chuỗi bằng -2; cộng 1 để mã trang luôn dương
            inp.write_bytes("".join(" ".join(f"{x + 1} -1" for x in s) + " -2\n" for s in db).encode("utf-8"))
            out = tmp / f"spmf_{n}_{r}.txt"
            pct = f"{float(r) * 100:g}%"
            subprocess.run([spmf_exe, "run", "PrefixSpan", str(inp), str(out), pct, "1", str(sw.MAX_LEN)],
                           check=True, capture_output=True, timeout=900)
            spmf = {}
            for line in out.read_text(encoding="utf-8").splitlines():
                if line.startswith("@") or not line.strip():
                    continue
                left, sup = line.split("#SUP:")
                spmf[tuple(int(x) - 1 for x in left.split() if x != "-1")] = int(sup)
            min_sup = sw.min_sup_from_ratio(r, len(db))
            g, p = both(db, min_sup)
            info[f"{n}/{r}"] = {"n": len(db), "min_sup": min_sup, "spmf_patterns": len(spmf),
                                "gsp_patterns": len(g), "prefixspan_patterns": len(p)}
            checks[f"n={len(db)}, r={pct} (min_sup={min_sup}): GSP == PrefixSpan == SPMF"] = g == p == spmf
    return checks, info


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--spmf", default=None)
    ap.add_argument("--data", default=str(HERE / "data" / "sequences.tsv"))
    args = ap.parse_args()
    sys.stdout.reconfigure(encoding="utf-8")
    (HERE / "ket-qua").mkdir(exist_ok=True)
    report = {"layer1": layer1(), "layer2": layer2(), "layer3": layer3()}
    if args.spmf:
        report["layer4"], report["layer4_info"] = layer4(args.spmf, args.data)
    ok = True
    for name in ("layer1", "layer2", "layer3", "layer4"):
        for k, v in report.get(name, {}).items():
            if k.startswith("_"):
                continue
            ok &= bool(v)
            print(f"[{'ĐẠT' if v else 'LỖI'}] {name}: {k}")
    report["all_passed"] = ok
    (HERE / "ket-qua" / "kiem_tra_tinh_dung.json").write_bytes(
        json.dumps(report, ensure_ascii=False, indent=2, default=str).encode("utf-8"))
    print("TẤT CẢ ĐẠT" if ok else "CÓ LỖI")
    return 0 if ok else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
r = subprocess.run([sys.executable, "kiem_tra_tinh_dung.py"], capture_output=True, text=True, encoding="utf-8")
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0

## 5. Đo thời gian và bộ nhớ

Mỗi lượt là một tiến trình Python mới. Thời gian đo bằng `time.perf_counter()` từ ngay trước khi đọc tệp chuỗi tới khi ghi xong tệp mẫu; bộ nhớ là đỉnh cấp phát Python (`tracemalloc`) đo ở lượt riêng. Mỗi cấu hình: 1 lượt khởi động, 3 lượt thời gian (lượt lẻ GSP chạy trước, lượt chẵn PrefixSpan chạy trước), 1 lượt bộ nhớ; giới hạn 600 giây mỗi lượt.

In [ ]:
%%writefile chay_do.py
"""Điều phối phép đo GSP vs PrefixSpan (mục 5.3–5.7 của bản nội dung).

Mỗi lượt là một tiến trình Python mới: python -I spm_wiki.py bench ...
    python -I chay_do.py thu      # chạy thử khả năng hoàn thành: 1 lượt thời gian mỗi cấu hình
    python -I chay_do.py chinh --timeout 600
                                  # đo chính: 1 khởi động + 3 lượt thời gian + 1 lượt bộ nhớ
Kết quả: ket-qua/<tên>/runs.jsonl, tệp mẫu đầu ra và machine.json.
"""
import argparse
import json
import os
import platform
import subprocess
import sys
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
SPM = HERE / "spm_wiki.py"
DATA = HERE / "data" / "sequences.tsv"
ALGOS = ("gsp", "prefixspan")

# (nhóm thí nghiệm, số chuỗi, r); cấu hình 20.000 / 0,2% dùng chung cho hai nhóm
FULL_N = None  # None = toàn bộ tập hợp lệ


def configs(full_n):
    out = [("nguong", 20000, r) for r in ("0.01", "0.005", "0.002", "0.001")]
    out += [("quy_mo", n, "0.002") for n in (5000, 10000, 40000, full_n)]
    return out


def machine_info():
    info = {"python": sys.version, "platform": platform.platform(), "cpu_count": os.cpu_count()}
    try:
        if sys.platform == "win32":
            info["cpu"] = subprocess.run(
                ["powershell", "-NoProfile", "-Command", "(Get-CimInstance Win32_Processor).Name"],
                capture_output=True, text=True, timeout=60).stdout.strip()
        else:
            info["cpu"] = next(line.split(":", 1)[1].strip() for line in open("/proc/cpuinfo")
                               if line.startswith("model name"))
    except Exception as e:  # thông tin phụ, không chặn phép đo
        info["cpu"] = f"unknown ({e})"
    try:
        import psutil
        info["ram_bytes"] = psutil.virtual_memory().total
    except Exception:
        info["ram_bytes"] = None
    return info


def run_one(outdir, group, n, r, algo, mode, label, timeout):
    tag = f"n{n}_r{r}_{algo}"
    out = outdir / "patterns" / f"{tag}.tsv"
    js = outdir / "tmp" / f"{tag}_{label}.json"
    cmd = [sys.executable, "-I", str(SPM), "bench", "--algo", algo, "--data", str(DATA), "--n", str(n),
           "--r", r, "--mode", mode, "--out", str(out), "--json", str(js)]
    t0 = time.perf_counter()
    try:
        subprocess.run(cmd, check=True, capture_output=True, timeout=timeout)
        rec = json.loads(js.read_text(encoding="utf-8"))
        rec["status"] = "ok"
    except subprocess.TimeoutExpired:
        rec = {"algo": algo, "n": n, "r": r, "mode": mode, "status": "timeout", "timeout_s": timeout}
    except subprocess.CalledProcessError as e:
        rec = {"algo": algo, "n": n, "r": r, "mode": mode, "status": "error",
               "stderr": e.stderr.decode("utf-8", "replace")[-2000:]}
    rec.update({"group": group, "label": label, "wall_process_s": time.perf_counter() - t0})
    with open(outdir / "runs.jsonl", "a", encoding="utf-8", newline="\n") as f:
        f.write(json.dumps(rec) + "\n")
    shown = f"{rec.get('seconds', 0):.2f}s" if rec["status"] == "ok" else rec["status"]
    peak = f" peak={rec['peak_bytes'] / 2**20:.1f}MB" if rec.get("peak_bytes") else ""
    print(f"{group:7s} n={n:6d} r={r:6s} {algo:10s} {label:8s} {shown}{peak} "
          f"patterns={rec.get('n_patterns', '-')}", flush=True)
    return rec


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["thu", "chinh"])
    ap.add_argument("--timeout", type=float, default=1800)
    ap.add_argument("--outdir", default=None)
    args = ap.parse_args()
    sys.stdout.reconfigure(encoding="utf-8")
    full_n = sum(1 for _ in open(DATA, encoding="utf-8")) - 1
    outdir = Path(args.outdir or HERE / "ket-qua" / ("chay_thu" if args.mode == "thu" else "do_chinh"))
    (outdir / "patterns").mkdir(parents=True, exist_ok=True)
    (outdir / "tmp").mkdir(exist_ok=True)
    meta = {"machine": machine_info(), "timeout_s": args.timeout, "mode": args.mode, "full_n": full_n,
            "started": time.strftime("%Y-%m-%d %H:%M:%S")}
    (outdir / "machine.json").write_bytes(json.dumps(meta, indent=2).encode("utf-8"))
    for group, n, r in configs(full_n):
        if args.mode == "thu":
            for algo in ALGOS:
                run_one(outdir, group, n, r, algo, "time", "thu", args.timeout)
            continue
        alive = {}
        for algo in ALGOS:  # lượt khởi động, không tính
            alive[algo] = run_one(outdir, group, n, r, algo, "time", "warmup", args.timeout)["status"] == "ok"
        for rep in (1, 2, 3):  # lượt lẻ GSP trước, lượt chẵn PrefixSpan trước
            order = ALGOS if rep % 2 == 1 else ALGOS[::-1]
            for algo in order:
                if alive[algo]:
                    run_one(outdir, group, n, r, algo, "time", f"time{rep}", args.timeout)
        for algo in ALGOS:
            if alive[algo]:
                run_one(outdir, group, n, r, algo, "mem", "mem", args.timeout)
    meta["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
    (outdir / "machine.json").write_bytes(json.dumps(meta, indent=2).encode("utf-8"))


if __name__ == "__main__":
    main()

In [ ]:
r = subprocess.run([sys.executable, "chay_do.py", "chinh", "--timeout", "600"], capture_output=True, text=True, encoding="utf-8")
print(r.stdout[-6000:], r.stderr[-2000:])
assert r.returncode == 0

In [ ]:
%%writefile tong_hop.py
"""Tổng hợp kết quả đo chính thành bảng và biểu đồ (mục 5.9 của bản nội dung).

    python -I tong_hop.py [--indir ket-qua/do_chinh]
Ghi: bang_ket_qua.md, tong_hop.json và các biểu đồ PNG trong cùng thư mục.
"""
import argparse
import json
import statistics
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

HERE = Path(__file__).resolve().parent
LABEL = {"gsp": "GSP", "prefixspan": "PrefixSpan"}
COLOR = {"gsp": "#c2410c", "prefixspan": "#1d4ed8"}


def load(indir):
    runs = [json.loads(line) for line in open(indir / "runs.jsonl", encoding="utf-8")]
    table = defaultdict(lambda: {"times": [], "status": []})
    for r in runs:
        key = (int(r["n"]), r["r"], r["algo"])
        row = table[key]
        row.setdefault("groups", set()).add(r["group"])
        if r["label"] == "warmup":
            row["warmup_status"] = r["status"]
        elif r["mode"] == "time":
            row["status"].append(r["status"])
            if r["status"] == "ok":
                row["times"].append(r["seconds"])
                row.update(min_sup=r["min_sup"], n_patterns=r["n_patterns"], sha=r["sha256"],
                           stats=r["stats"], by_length=r["by_length"])
        elif r["mode"] == "mem" and r["status"] == "ok":
            row["peak_mb"] = r["peak_bytes"] / 2 ** 20
            row["mem_seconds"] = r["seconds"]
    return table


def summarize(table):
    configs = sorted({(n, r) for n, r, _ in table}, key=lambda t: (t[0], -float(t[1])))
    rows = []
    for n, r in configs:
        g, p = table.get((n, r, "gsp")), table.get((n, r, "prefixspan"))
        entry = {"n": n, "r": r, "groups": sorted(set().union(*(x.get("groups", set()) for x in (g, p) if x)))}
        for name, x in (("gsp", g), ("prefixspan", p)):
            ok = x and x["times"] and all(s == "ok" for s in x["status"])
            entry[name] = {
                "status": "ok" if ok else (x.get("warmup_status") or "missing") if x else "missing",
                "times": x["times"] if x else [],
                "median": statistics.median(x["times"]) if ok else None,
                "min": min(x["times"]) if ok else None, "max": max(x["times"]) if ok else None,
                "peak_mb": x.get("peak_mb") if x else None,
                "n_patterns": x.get("n_patterns") if x else None,
                "min_sup": x.get("min_sup") if x else None,
                "sha": x.get("sha") if x else None,
                "stats": x.get("stats") if x else None,
                "by_length": x.get("by_length") if x else None,
            }
        both_ok = entry["gsp"]["status"] == entry["prefixspan"]["status"] == "ok"
        entry["outputs_identical"] = both_ok and entry["gsp"]["sha"] == entry["prefixspan"]["sha"]
        entry["R_T"] = entry["gsp"]["median"] / entry["prefixspan"]["median"] if both_ok else None
        entry["R_M"] = (entry["gsp"]["peak_mb"] / entry["prefixspan"]["peak_mb"]
                        if both_ok and entry["prefixspan"]["peak_mb"] else None)
        rows.append(entry)
    return rows


def vn(x, digits=0):
    """Định dạng số kiểu Việt Nam: dấu chấm hàng nghìn, dấu phẩy thập phân."""
    if x is None:
        return "—"
    s = f"{x:,.{digits}f}"
    return s.replace(",", "_").replace(".", ",").replace("_", ".")


def vn_ratio(r):
    return vn(float(r) * 100, 1 if float(r) * 1000 % 10 else 0) + "%"


def markdown(rows):
    head = ("| \\|D\\| | r | min_sup | Thuật toán | Thời gian 3 lượt (s) | Trung vị (s) | Đỉnh cấp phát Python (MB) "
            "| Số mẫu | Ứng viên GSP mức 2 / 3 / 4 | Đầu ra trùng | R_T | R_M |\n|---|---|---|---|---|---|---|---|---|---|---|---|")
    lines = [head]
    for e in rows:
        for name in ("gsp", "prefixspan"):
            x = e[name]
            cand = "—"
            if name == "gsp" and x["stats"]:
                c = x["stats"]["candidates"]
                cand = " / ".join(vn(c.get(str(k), c.get(k, 0))) for k in (2, 3, 4))
            times = "; ".join(vn(t, 3) for t in x["times"]) if x["status"] == "ok" else x["status"]
            same = ("có" if e["outputs_identical"] else "KHÔNG") if name == "gsp" else ""
            rt = vn(e["R_T"], 2) if name == "gsp" else ""
            rm = vn(e["R_M"], 2) if name == "gsp" else ""
            lines.append(f"| {vn(e['n'])} | {vn_ratio(e['r'])} | {vn(x['min_sup'])} | {LABEL[name]} | {times} | "
                         f"{vn(x['median'], 3)} | {vn(x['peak_mb'], 1)} | {vn(x['n_patterns'])} | {cand} | "
                         f"{same} | {rt} | {rm} |")
    return "\n".join(lines) + "\n"


def plot(rows, outdir):
    by_r = [e for e in rows if "nguong" in e["groups"]]
    by_n = [e for e in rows if "quy_mo" in e["groups"] or (e["n"] == 20000 and e["r"] == "0.002")]
    by_r.sort(key=lambda e: -float(e["r"]))
    by_n.sort(key=lambda e: e["n"])
    specs = [
        (by_r, lambda e: vn_ratio(e["r"]), "median", "Thời gian trung vị (s)",
         "Thời gian theo ngưỡng r (|D| = 20.000, độ dài ≤ 4)", "thoi_gian_theo_nguong.png", "Ngưỡng r"),
        (by_n, lambda e: vn(e["n"]), "median", "Thời gian trung vị (s)",
         "Thời gian theo số chuỗi (r = 0,2%, độ dài ≤ 4)", "thoi_gian_theo_quy_mo.png", "Số chuỗi |D|"),
        (by_r, lambda e: vn_ratio(e["r"]), "peak_mb", "Đỉnh cấp phát Python (MB)",
         "Bộ nhớ theo ngưỡng r (|D| = 20.000, độ dài ≤ 4)", "bo_nho_theo_nguong.png", "Ngưỡng r"),
        (by_n, lambda e: vn(e["n"]), "peak_mb", "Đỉnh cấp phát Python (MB)",
         "Bộ nhớ theo số chuỗi (r = 0,2%, độ dài ≤ 4)", "bo_nho_theo_quy_mo.png", "Số chuỗi |D|"),
    ]
    for data, xl, key, ylabel, title, fname, xlabel in specs:
        fig, ax = plt.subplots(figsize=(7, 4.2), dpi=150)
        # trục số chuỗi theo tỷ lệ thật; trục ngưỡng r là các mức rời rạc
        xs = [e["n"] for e in data] if data is by_n else list(range(len(data)))
        for name in ("gsp", "prefixspan"):
            ys = [e[name][key] for e in data]
            ax.plot(xs, ys, marker="o", color=COLOR[name], label=LABEL[name], linewidth=2)
            if key == "median":
                lo = [e[name]["min"] for e in data]
                hi = [e[name]["max"] for e in data]
                ax.fill_between(xs, lo, hi, color=COLOR[name], alpha=0.15, linewidth=0)
        ax.set_xticks(xs, [xl(e) for e in data])
        if data is by_n:
            plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontsize=11)
        ax.set_ylim(bottom=0)
        ax.grid(axis="y", alpha=0.3)
        ax.legend(frameon=False)
        fig.tight_layout()
        fig.savefig(outdir / fname)
        plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--indir", default=str(HERE / "ket-qua" / "do_chinh"))
    args = ap.parse_args()
    sys.stdout.reconfigure(encoding="utf-8")
    indir = Path(args.indir)
    rows = summarize(load(indir))
    md = markdown(rows)
    (indir / "bang_ket_qua.md").write_bytes(md.encode("utf-8"))
    (indir / "tong_hop.json").write_bytes(json.dumps(rows, ensure_ascii=False, indent=2).encode("utf-8"))
    plot(rows, indir)
    print(md)


if __name__ == "__main__":
    main()

In [ ]:
r = subprocess.run([sys.executable, "tong_hop.py"], capture_output=True, text=True, encoding="utf-8")
assert r.returncode == 0, r.stderr
from IPython.display import Markdown, Image, display
display(Markdown(r.stdout))
for name in ["thoi_gian_theo_nguong", "thoi_gian_theo_quy_mo", "bo_nho_theo_nguong", "bo_nho_theo_quy_mo"]:
    display(Image(filename=f"ket-qua/do_chinh/{name}.png"))

## 6. Đánh giá gợi ý trên tập test

Train = ⌊0,8·|D|⌋ đường đi đầu, test = phần còn lại. Mẫu khai phá trên train bằng cả hai thuật toán và chỉ dùng khi hai tập (mẫu, sup) trùng nhau. Quy tắc gợi ý cố định: ngữ cảnh 3 → 2 → 1 trang cuối, ⟨h,y⟩ phải là mẫu phổ biến, C(h) ≥ 5, C(h,y) ≥ 2, xếp theo C(h,y)/C(h). Baseline: ba trang có nhiều lượt ghé nhất trong train, bỏ trang hiện tại.

In [ ]:
%%writefile danh_gia.py
"""Đánh giá hữu ích trên tập test (mục 4.7 và 5.10 của bản nội dung).

Train = floor(0,8·|D|) đường đi đầu theo thứ tự đã xáo trộn, test = phần còn lại.
Mẫu khai phá trên train bằng cả hai thuật toán; chỉ dùng khi hai tập (mẫu, sup) trùng nhau.
    python -I danh_gia.py [--ratios 0.002 0.01 0.005 0.001] [--algos gsp prefixspan]
Kết quả: ket-qua/danh_gia/danh_gia.json và ca_minh_hoa.md.
"""
import argparse
import json
import sys
import time
from collections import Counter
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
import spm_wiki as sw  # noqa: E402

DATA = HERE / "data"


def ratio(a, b):
    return {"num": a, "den": b, "value": a / b if b else None}


def summarize(raw):
    out = {}
    for g, c in raw["groups"].items():
        T = c["T"]
        res = {"T": T}
        for name in ("baseline", "pattern", "pattern+fill"):
            res[name] = {"coverage": ratio(c[f"{name}.cov"], T), "hit1": ratio(c[f"{name}.hit1"], T),
                         "hit3": ratio(c[f"{name}.hit3"], T)}
        res["covered_subset"] = {"T_cov": c["cov.T"],
                                 "pattern_hit3": ratio(c["cov.pattern.hit3"], c["cov.T"]),
                                 "baseline_hit3": ratio(c["cov.baseline.hit3"], c["cov.T"])}
        out[g] = res
    out["context_len"] = raw["context_len"]
    return out


def demo_cases(rec, test, backs, sids, names):
    """Ca đầu tiên theo thứ tự test cho mỗi loại: gợi ý trúng, gợi ý trượt, không đủ bằng chứng."""
    found = {}
    for i, t, hist, ans, forward in sw.test_cases(test, backs):
        ctx, ranked = rec.explain(hist)
        kind = "khong_du_bang_chung" if not ranked else (
            "trung" if ans in [e["page"] for e in ranked] else "truot")
        if kind in found:
            continue
        tried = []
        for m in range(min(rec.max_context, len(hist)), 0, -1):
            h = tuple(hist[-m:])
            n_ok = sum(1 for y, chy in rec.c_hy[h].items()
                       if chy >= rec.min_transition and h + (y,) in rec.patterns)
            tried.append({"context": [names[p] for p in h], "c_h": rec.c_h.get(h, 0),
                          "qualified_candidates": n_ok if rec.c_h.get(h, 0) >= rec.min_context else 0})
        found[kind] = {
            "sid": sids[i], "t": t, "history": [names[p] for p in hist], "answer": names[ans],
            "answer_is_forward_click": forward, "context": [names[p] for p in ctx], "tried": tried,
            "suggestions": [dict(e, page=names[e["page"]]) for e in ranked],
        }
        if len(found) == 3:
            break
    return found


def example_patterns(patterns, names, top=10):
    """Mẫu tiêu biểu theo quy tắc cố định: sup cao nhất theo độ dài; cặp hai trang khác nhau kèm sup
    chiều ngược; mẫu quay về dạng x→y→x với x ≠ y; mẫu lặp một trang x→x."""
    def fmt(p):
        return " → ".join(names[x] for x in p)

    def ranked(cond):
        return sorted(((c, p) for p, c in patterns.items() if cond(p)), key=lambda t: (-t[0], t[1]))[:top]
    out = {}
    for k in (2, 3, 4):
        out[f"len{k}"] = [{"pattern": fmt(p), "sup": c} for c, p in ranked(lambda p, k=k: len(p) == k)]
    out["order"] = [{"pattern": fmt(p), "sup": c, "reverse_sup": patterns.get(p[::-1], 0)}
                    for c, p in ranked(lambda p: len(p) == 2 and p[0] != p[1])]
    out["return_xyx"] = [{"pattern": fmt(p), "sup": c}
                         for c, p in ranked(lambda p: len(p) == 3 and p[0] == p[2] != p[1])]
    out["repeat_xx"] = [{"pattern": fmt(p), "sup": c} for c, p in ranked(lambda p: len(p) == 2 and p[0] == p[1])]
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ratios", nargs="+", default=["0.002", "0.01", "0.005", "0.001"])
    ap.add_argument("--algos", nargs="+", default=["gsp", "prefixspan"])
    args = ap.parse_args()
    sys.stdout.reconfigure(encoding="utf-8")
    outdir = HERE / "ket-qua" / "danh_gia"
    outdir.mkdir(parents=True, exist_ok=True)
    info = json.loads((DATA / "prepare_info.json").read_text(encoding="utf-8"))
    n_train = info["n_train"]
    names = sw.read_names(DATA / "articles.tsv")
    train = sw.read_sequences(DATA / "sequences.tsv", n_train)
    test, backs, sids = sw.read_sequences(DATA / "sequences.tsv", None, start=n_train, with_back=True)
    report = {"n_train": len(train), "n_test": len(test),
              "rules": {"max_context": sw.MAX_CONTEXT, "min_C_h": sw.MIN_CONTEXT_COUNT,
                        "min_C_hy": sw.MIN_TRANSITION_COUNT, "max_len": sw.MAX_LEN},
              "by_ratio": {}}
    for r in args.ratios:
        min_sup = sw.min_sup_from_ratio(r, len(train))
        mined, seconds = {}, {}
        for algo in args.algos:
            t0 = time.perf_counter()
            mined[algo] = sw.ALGORITHMS[algo](train, min_sup)[0]
            seconds[algo] = time.perf_counter() - t0
        first = mined[args.algos[0]]
        same = all(m == first for m in mined.values())
        if not same:
            raise SystemExit(f"r={r}: hai tập mẫu khác nhau, dừng đánh giá")
        rec = sw.Recommender(train, first)
        t0 = time.perf_counter()
        res = summarize(sw.evaluate(rec, test, backs))
        res.update({"min_sup": min_sup, "n_patterns": len(first),
                    "by_length": dict(Counter(len(p) for p in first)),
                    "algos_checked": args.algos, "outputs_identical": same,
                    "mining_seconds_untimed_context": seconds,
                    "eval_seconds": time.perf_counter() - t0,
                    "baseline_pages": [names[p] for p in rec.popular[:4]]})
        if r == args.ratios[0]:
            res["demo_cases"] = demo_cases(rec, test, backs, sids, names)
            res["example_patterns"] = example_patterns(first, names)
        report["by_ratio"][r] = res
        a = res["all"]
        print(f"r={r} min_sup={min_sup} patterns={len(first)} identical={same} | "
              f"T={a['T']} cov={a['pattern']['coverage']['value']:.4f} "
              f"hit3 pattern={a['pattern']['hit3']['value']:.4f} fill={a['pattern+fill']['hit3']['value']:.4f} "
              f"baseline={a['baseline']['hit3']['value']:.4f}", flush=True)
    (outdir / "danh_gia.json").write_bytes(json.dumps(report, ensure_ascii=False, indent=2).encode("utf-8"))


if __name__ == "__main__":
    main()

In [ ]:
r = subprocess.run([sys.executable, "danh_gia.py"], capture_output=True, text=True, encoding="utf-8")
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0
res = json.loads(Path("ket-qua/danh_gia/danh_gia.json").read_text(encoding="utf-8"))
main = res["by_ratio"]["0.002"]
for g in ("all", "forward"):
    x = main[g]
    print(f"\n== {g}: T = {x['T']}")
    for name in ("baseline", "pattern", "pattern+fill"):
        m = x[name]
        print(f"{name:13s} coverage={m['coverage']['value']:.4f}  Hit@1={m['hit1']['value']:.4f}  Hit@3={m['hit3']['value']:.4f}")
    c = x["covered_subset"]
    print(f"Trên {c['T_cov']} ca có gợi ý: Hit@3 mẫu={c['pattern_hit3']['value']:.4f}, baseline={c['baseline_hit3']['value']:.4f}")

In [ ]:
for kind, case in main["demo_cases"].items():
    print(f"\n### {kind}")
    print("Lịch sử:", " → ".join(case["history"]))
    print("Đáp án:", case["answer"], "| click tiến:", case["answer_is_forward_click"])
    print("Ngữ cảnh dùng:", " → ".join(case["context"]) or "(không có)")
    for e in case["suggestions"]:
        print(f"  {e['page']}: C(h)={e['c_h']}, C(h,y)={e['c_hy']}, điểm={e['score']:.3f}, sup={e['sup']}")